day02

0. 복습
회귀분석
 : 변수들 사이의 관계를 찾아내어, 숫자를 예측하는 것

선형 회귀
 : 변수들 사이의 관계를 직선으로 나타내는 회귀

단순 선형 회귀
 :한개의 특성으로 타깃을 예측할 수 있는 회귀 분석
ex) 자동차의 무게 -> 연비
	y = ax + b


1.다중 선형 회귀
 : 특성(입력)을 여러개 써서 타깃(숫자)을 예측하는 선형 회귀
- 단순 선형 회귀는 특성이 하나라 식이 y = ax + b 였다
- 다중 선형 회귀는 특성이 여러개이므로, 각 특성마다 기울기(계수)가
  붙는다
	y = a1 * x1 + a2 * x2 + ... + b
- 각 기울기(계수)는 "그 특성이 1늘어날 때 y가 얼마나 편하는지"
- b(절편)는 모든 특성이 0일때의 기준값
ex) 시험 점수를 공부시간 하나로 예측하는게 단순 회귀라면,
    공부시간 + 수면 시간 + 출석을 함께 넣어 예측하는 것이
    다중 회귀이다

1) 회귀 분석의 가정
- 선형 회귀는 아무 데이터나 잘 통하는 것이 아닌, 몇 가지 전제(가정)
  위에서 동작한다
(1) 선형성 : x와 y의 관계가 대체로 직선 모양
(2) 독립성 : 각 데이터의 오차가 서로 무관
	앞 예측 실수가 뒤에 영향 주면 안된다
(3) 등분산성 : 오차의 퍼짐이 어디서나 비슷
	예측이 큰 구간만 유독 많이 틀리면 위반
(4) 정규성 : 오차가 0을 중심으로 정규분포
	대부분 오차가 작고, 오차가 큰 건 들물어야
(5) 다중공선성이 없음 : 특성끼리 너무 비슷하면 안된다
			=> 큰 상관관계를 가지면 좋지 않다
			(다중 회귀 전용)
		ex) 무게, 배기량 처럼 거의 같은 정보면 중복
** 다중공선성이 문제인 이유
- 무게와 배기량이 거의 똑같이 움직이면, 모델은 "연비가 낮은게
  무게 때문인지 배기량 때문인지" 헷갈린다 => 각 계수의 값이
  불안정해진다(어떤 특성이 얼마나 중요한지)


2. 다항 회귀
 : 데이터의 관계가 곡선일때, 직선 대신 곡선으로 맞추는 회귀
- 선형 회귀는 y=ax + b, 즉 직선만 그릴 수 있다
  => 관계에 곡선이 있으면 직선으론 한계가 있다
- 다항 회귀는 x^2, x^3 같은 제곱, 세제곱 항을 더해 곡선으로 표현
	y = a*x^2 + a*x + b	<- 2차
	y = a*x^3 + a*x^2 + a*x + b	<- 3차(더 복잡한 곡선)
- 몇 제곱까지 쓰느냐를 차수(degree)라고 한다
- 차수가 높을수록 더 복잡한 곡선을 그릴 수 있다

1) 차수를 높이면 => 과적합 주의
- "차수를 높일수록 곡선이 유연해지니, 무조건 높이면 좋은 것 아닐까?"
=> 아니다. 차수가 너무 높으면 과적합(overfitting)이 발생할 수 있다.
** 과적합(과대적합) : 모델이 훈련데이터(train)만 너무 맞춰서
		      정작 새로운 데이터에 대한 예측력이 낮아지는 것

2) 적정 차수 찾기
- 평가용 R^2가 가장 높은 차수를 고르면 된다.
  (예제에서는 2 ~ 5차 정도가 좋음)
- 너무 높은 차수는 과적합이라 피한다
- "평가용 성능이 가장 좋으면서, 되도록 낮은(단순한) 차수"를 고른다
- 보통은 2차, 3차 정도가 적당하다

## 다중 선형 회귀

In [ ]:
import pandas as pd
import seaborn as sns

# 다중 선형 회귀 분석
# : 무게, 마력, 연식 3개의 특성으로 연비를 예측하는 다중 선형 회귀
mpg = sns.load_dataset("mpg")

mpg.head()

In [ ]:
# 사용할 데이터에 결측치 확인 및 처리
print(mpg[['mpg', 'weight', 'horsepower', 'model_year']].isnull().sum())

# 마력(horsepower)이 결측치인 행을 제거
mpg = mpg.dropna(subset=['mpg', 'weight', 'horsepower', 'model_year'])
print(mpg[['mpg', 'weight', 'horsepower', 'model_year']].isnull().sum())

### 특성과 타깃 나누기

In [ ]:
# 특성 X : 무게, 마력, 연식
features = ['weight', 'horsepower', 'model_year']
X = mpg[features]

# 타깃 y : 연비
y = mpg['mpg']

In [ ]:
# 모델을 만들기 전에, 특성들끼리 너무 비슷(다중 공선성)하지 않은지 확인
# => 상관관계로 확인

X.corr().round(2)
# 무게와 마력(horsepower)의 상관계수가 0.86으로 상당히 높다
# => 다중공선성 문제가 있을 수 있다

In [ ]:
# 다중 선형 회귀 모델 생성 + 학습 + 평가
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression # 선형 회귀 모델 생성
from sklearn.metrics import r2_score # 결정 계수(모델 설명력)

# 특성 조합을 바꾸면서 결정계수 비교
def check_r2(cols) :
    Xc = mpg[cols] # 특성
    X_train, X_test, y_train, y_test = train_test_split(Xc, y,
                                                       test_size=0.2,
                                                       random_state=42)
    # 모델 생성후 학습
    model = LinearRegression().fit(X_train, y_train)
    return r2_score(y_test, model.predict(X_test)) # 예측 후 결정계수 반환

# 특성 조합별 결과 확인
print(f"셋다(무게 + 마력 + 연식) : {check_r2(['weight', 'horsepower', 'model_year']) : .3f}")
# 무게와 마력이 정보가 겹치므로 둘 중 하나를 제외
print(f"마력 제거 : {check_r2(['weight', 'model_year']) : .3f}")
print(f"무게 제거 : {check_r2(['horsepower', 'model_year']) : .3f}")
# 마력을 빼도 결정 계수(R^2)가 크게 변하지는 않는다(살짝 오름)
# => 마력의 정보가 이미 무게 안에 거의 들어 있어서, 마력을 빼도 손해가 없다
# => 겹치던 특성 하나를 제거해 다중공선성 문제 해결
# 반대로, 무게를 빼면 0.676으로 뚝 떨어진다. 무게가 마력보다는 연비를
# 더 잘 설명하는 핵심 특성이라는 뜻이다
# 결론 : 겹치는 특성 중 정보가 약한 마력을 제거하고 무게를 남기면,
#        성능은 유지하면서 모델이 더 단순해지고 계수도 안정적이 된다

### 학습 / 평가 분할 + 모델 학습

In [ ]:
# 방법은 단순 회귀와 완전히 같다

# 8 : 2로 훈련/테스트 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"학습용 : {len(X_train)}, 평가용 : {len(X_test)}")

# 모델 학습
model = LinearRegression()
model.fit(X_train, y_train) # 훈련 데이터로 모델 학습

### 계수 해석 - 특성별 기울기

In [ ]:
import numpy as np

coef_table = pd.DataFrame({
    "특성" : features,
    "계수(기울기)" : np.round(model.coef_, 4)
})
# weight 계수 -0.0065 : 다른 조건이 같을 때, 무게가 무거울수록 연비가 낮아진다
# horsepower 계수 -0.0088 : 마력이 높을수록 연비가 낮아진다
# model_year 계수 +0.7594 : 연식이 1년 최신일수록 연비가 0.76 높아진다
# => 다중 회귀는 "각 요인이 결과에 어느 방향으로 얼마나 기여하는지"확인 가능
coef_table

## 예측 + 평가

In [ ]:
# 새로운 자동차의 연비를 예측하고, 평가용 데이터로 성능 평가

# 무게 3000, 마력 100, 76년식 자동차의 연비 예측
new_car = pd.DataFrame({"weight" : [3000], "horsepower" : [100],
                       "model_year" : [76]})
print(f"연비 예측 : {model.predict(new_car)[0]}")

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

# 평가용으로 모델 성능 측정
pred_test = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_test)) # 실제데이터, 예측
r2 = r2_score(y_test, pred_test)
print(f"RMSE : {rmse : .3f}")
print(f"R^2 : {r2 : .3f}")
# RMSE : 3.255 => 예측이 실제 연비와 평균적으로 약 3.3 차이
# R^2 : 0.792 => 3개의 특성(마력, 무게, 연식)이 연비의 변동의 약 79%를 설명한다

In [ ]:
# 단순 vs 다중
# R^2가 0.723(day01) -> 0.792로 올랐다
# 마력과 연식이라는 관련 특성을 더 넣으니, 모델이 설명하는 폭이 넓어졌다

### 다항 회귀

In [ ]:
# 다항 회귀의 원리는 의외로 단순
#  x^2, x^3 을 새로운 특성을 추가하는 것 뿐이다
# - 선형 회귀에서 특성이 x하나였다면, x^2, x^3을 또 하나의 특성으로
#   만들어 넣는다
# - 이러한 특성 확장을 할때, scikit-learn의 polynomaialFeatures을 사용한다

from sklearn.preprocessing import PolynomialFeatures

# degree=2 : 각 값을 [x, x^2]로 확장
# include_bias = False : 상수항(1로만 된 열)은 빼고 본다
poly = PolynomialFeatures(degree=2, include_bias=False)
print(poly.fit_transform([[3], [5]]))
# x^2열이 새로 생긴 것 뿐이다

In [ ]:
import pandas as pd
import seaborn as sns

# 자동차 연비 데이터에서 마력 -> 연비의 관계를 보는 다항회귀
mpg = sns.load_dataset('mpg').dropna(subset=['horsepower', 'mpg'])

# 특성 / 타깃 분리
X = mpg[['horsepower']] # 특성 : 마력
y = mpg['mpg'] # 타깃 : 연비

In [ ]:
# 회귀 분석 전에 관계 확인(산점도 시각화)
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(6, 4))
plt.scatter(mpg['horsepower'], mpg['mpg'], alpha=0.4)
plt.xlabel("마력")
plt.ylabel("연비")
plt.title("마력과 연비의 관계")
plt.show()
# 점들이 직선이 아닌 곡선의 모양으로 이어져 있다
# 마력이 낮을 땐 연비가 가파르게 떨어지다가,
# 마력이 높아지면 완만해진다 => 직선 하나로는 이 곡선을 표현하기 어렵다

### 선형(직선) 회귀의 한계

In [ ]:
# 단순 선형 회귀로 마력과 연비의 관계 확인
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 1차 = 선형회귀
liner = LinearRegression().fit(X_train, y_train)
pred = liner.predict(X_test)
print(f"직선 R^2 : {r2_score(y_test, pred) : .3f}")
print(f"직선 RMSE : {np.sqrt(mean_squared_error(y_test, pred)) : .3f}")

# 단순 선형 회귀 모델의 설명력이 0.566으로 나쁘진 않지만
# 곡선 데이터를 직선으로 맞췄으니 한계가 있

## 다항 회귀(2차)

In [ ]:
# PolynomialFeatures로 x^2을 추가한뒤, 선형 회귀를 하면된다
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# PolynomialFeatures(2) : x -> [x, x^2]로 확장
# poly = PolynomialFeatures(2, include_bias= False)
# poly_x = poly.fit_transform(X_train)
# model = LinearRegression()
# model.fit(poly_x, y_train)

# make_pipeline으로 위 과정을 하나로 묶을 수 있다
poly_model = make_pipeline(PolynomialFeatures(2, include_bias=False),
                          LinearRegression())
poly_model.fit(X_train, y_train) # 훈련 데이터로 모델 학습

# 테스트 데이터로 예측
pred2 = poly_model.predict(X_test)
# 예측 데이터로 모델 평가
print(f"2차 곡선 R^2 : {r2_score(y_test, pred2) : .3f}")
print(f"2차 곡선 RMSE : {np.sqrt(mean_squared_error(y_test, pred2)) : .3f}")

# 직선으로 했을때보다 R^2이 올랐고, RMSE도 줄었다

## 모델 차수를 높이면 => 과대적합

In [ ]:
from sklearn.preprocessing import StandardScaler # 스케일링
# 차수를 아주 높이면 값이 폭발적으로 커져서 계산이 불안정해짐
# 그래서, 스케일리으로 단위를 맞춘다

for d in [1, 2, 5, 10, 15, 20] :
    # 스케일링 -> 특성확장(d차) -> 회귀를 하나로 묶기
    m = make_pipeline(StandardScaler(), PolynomialFeatures(d, include_bias=False),
                     LinearRegression())
    m.fit(X_train, y_train)
    r2_train = r2_score(y_train, m.predict(X_train)) # 학습용 데이터에 대한 성능
    r2_test = r2_score(y_train, m.predict(X_train))
    # 평가용 데이터에 대한 선능
    print(f"차수={d} | 학스용 R^2={r2_train : .3f} | 평가용 R^2={r2_test : .3f}")

# 학습용 결정계수는 높아질수록 계속 오른다(배운 데이터는 점점 더 잘 맞힌다)
# 하지만, 평가용 결정계수는 5차수 근처에서 최고를 찍고, 그 뒤로 오히려
# 떨어진다 => 처음보는 데이터는 더 못 맞히게 된다
# ** "학습용은 좋아지는데 평가용은 나빠지는"벌어짐이 바로 과적합이다
#    곡선이 학습 데이터의 점 하나 하나에 억지로 맞추려다, 전체 경향에서 벗어나 버린다

In [ ]:
# 1차(직선) vs 2차(곡선) vs 15차수(곡선) 시각화 비교

# 15차수(과적합) 모델 준비 - 고차라 스케일링 포함
overfit = make_pipeline(StandardScaler(), PolynomialFeatures(15, include_bias=False),
                       LinearRegression())
overfit.fit(X_train, y_train)

# 곡선을 부드럽게 나오게 하기 위한 촘촘한 마력값(최소 ~ 최대 사이 300개)
xs = pd.DataFrame({"horsepower" : np.linspace(X['horsepower'].min(),
                                             X['horsepower'].max(), 300)})
plt.figure(figsize=(7, 5))
plt.scatter(X_test["horsepower"], y_test, alpha=0.4, label="실제값(평가용)")
plt.plot(xs["horsepower"], liner.predict(xs),     color="green",  linewidth=2, label="1차(직선)")
plt.plot(xs["horsepower"], poly_model.predict(xs), color="red",    linewidth=2, label="2차(곡선)")
plt.plot(xs["horsepower"], overfit.predict(xs),    color="purple", linewidth=1, linestyle="--", label="15차(과적합)")
plt.ylim(5, 50)   # 15차가 위아래로 크게 튀므로 보기 좋게 y축 범위 제한
plt.xlabel("마력(horsepower)")
plt.ylabel("연비(mpg)")
plt.title("차수별 회귀선 비교")
plt.legend()
plt.show()
# 1차  : 곡선을 못 따라가 데이터에서 벗어난다(과소적합)
# 2차 : 전체 경향을 잘 따라간다(적합o)
# 15차 : 데이터가 작은 양 끝에서 미친듯이 요동친다
#        새로운 데이터에 대해서는 엉터리로 예측을 하는 과적합이 발생하고 있다

In [ ]:
# 2차 모델로 새로운 자동차의 연비 예측
new_car = pd.DataFrame({'horsepower': [100]})
print(f"마력이 100일때 예측 연비 : {poly_model.predict(new_car)[0] : .2f}")
# 곡선 위에서 마력 100에 해당하는 연비를 예측함

#### 과제

# 다중·다항 회귀 과제

이번엔 회귀를 두 방향으로 확장한다. **여러 특성을 함께 쓰는 다중 회귀**(문제 1)와, **곡선 관계를 맞추는 다항 회귀**(문제 2)다.
각 문제는 `데이터 준비 → 8:2 분할 → 모델 학습 → 지표(R²·RMSE) → 결과 해석 → 시각화` 흐름을 따른다.

## 문제 1) 펭귄의 여러 신체 치수로 체중 예측하기 (단순 vs 다중)

day01에서는 **날개 길이 하나**로 펭귄 체중을 예측했다. 이번엔 **세 가지 치수**를 함께 써서, 특성을 늘리면 예측이 좋아지는지 확인해보자.

- 특성: `bill_length_mm`(부리 길이), `bill_depth_mm`(부리 두께), `flipper_length_mm`(날개 길이)
- 타깃: `body_mass_g`(체중)

1. `penguins`를 불러와 위 네 열의 **결측치를 제거**한다.
2. 세 특성으로 `X`, 체중을 `y`로 두고 **8:2 분할**(`random_state=42`) 후 **다중 회귀** 학습.
3. **특성별 계수**를 표로 출력하고, **R²·RMSE**(평가용)를 출력한다.
4. **날개 길이 하나만** 쓴 단순 회귀의 R²와 **비교**한다. (특성을 늘리니 좋아졌는가?)
5. 부리 길이 45, 부리 두께 17, 날개 200인 펭귄의 체중을 **예측**한다.
6. **실제값 vs 예측값 산점도**를 그린다. (점들이 대각선에 가까울수록 잘 맞음)
7. 결과를 **해석**한다. (계수 부호·크기 / 단순과 비교해 얼마나 좋아졌는지 / 왜 그런지)

In [ ]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = "Malgun Gothic"
plt.rcParams['axes.unicode_minus'] = False # 마이너스(-) 깨짐방지


pg = sns.load_dataset("penguins")
# pg.head()

cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']
df_clean = pg.dropna(subset=cols)
# print(df_clean.isnull().sum())
X = df_clean[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']]
y = df_clean['body_mass_g']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

# 특성별 계수 표 출력
coef_table = pd.DataFrame({
    "특성" : X.columns,
    "기울기" : np.round(model.coef_, 4)
})
print(coef_table)
r2 = r2_score(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))

print(f"R^2 : {r2 : .2f}")
print(f"RMSE : {rmse : .2f}")

X_simple_train = X_train[['flipper_length_mm']]
X_simple_test = X_test[['flipper_length_mm']]

model_simple = LinearRegression()
model_simple.fit(X_simple_train, y_train)

pred_simple = model_simple.predict(X_simple_test)

r2_simple = r2_score(y_test, pred_simple)
rmse_simple = np.sqrt(mean_squared_error(y_test, pred_simple))

simpe_table = pd.DataFrame({
    "모델": ["단순 회귀 (날개 1개)", "다중 회귀 (3개 모두)"],
    "R²_simple (설명력)": [np.round(r2_simple, 4), np.round(r2, 4)],  # R² 수치 반올림
    "RMSE_simple (오차, g)": [np.round(rmse_simple, 4), np.round(rmse, 4)]
})
print(simpe_table)

new_pg = pd.DataFrame({
    'bill_length_mm' : [45],
    'bill_depth_mm' : [17],
    'flipper_length_mm' : [200]
})
new_pred = model.predict(new_pg)
print(f"새로운 펭귄 체중 예측 : {new_pred[0] : .2f}")

plt.figure(figsize=(6, 4))
plt.scatter(y_test, pred, alpha=0.5, color="red", label="실제 vs 예측")

xs = np.linspace(y_test.min(), y_test.max(), 300)

plt.plot(xs, xs, color="red", linewidth=2, label="예측선")
plt.xlabel("실제 체중(body_mass_g)")
plt.ylabel("예측 체중(pred)")
plt.title("실제값 vs 예측값 산점도 (다중 회귀)")
plt.legend()
plt.show()

# 부리길이 : 1mm 길어질수록 체중 4.01 증가
# 부리두께 : 1mm 두꺼워질수록 체중 10.9 증가
# 날개 길이 : 1mm 길어질수록 체중 49증
# 약간의 설명력 향상, 5정도의 오차가 줄음
# 예측 성능이 더 좋아졌으며, 다중 신체 정보 반영으로 바뀜, 오차 보완으로 단일 특성 때보다 신체를 구조적으로 파악할 수 있어서 좋아짐

## 문제 2) 재배 온도로 작물 수확량 예측하기 (직선 vs 곡선)

**`day02_작물수확.csv`**(작물 재배 실험 300건, 과제 노트북과 같은 폴더)로 다항 회귀를 해보자.

- 특성: `온도`(재배 온도, °C) / 타깃: `수확량`(kg)
- 너무 춥거나 너무 더우면 수확량이 낮고, **적당한 온도에서 가장 높다.** 즉 관계가 직선이 아니라 **곡선**일 수 있다. 직선 회귀와 다항 회귀 중 무엇이 맞는지 확인하자.

1. CSV를 불러와 `온도`(x) - `수확량`(y) **산점도**를 그려 관계 모양을 확인한다.
2. **8:2 분할**(`random_state=42`) 후, **직선(1차)** 과 **2차 다항 회귀**를 각각 학습해 **R²·RMSE를 비교**한다.
   - 힌트: `make_pipeline(PolynomialFeatures(2, include_bias=False), LinearRegression())`
3. **차수를 1·2·5·10**으로 바꿔가며 **학습용·평가용 R²**를 비교한다. (차수를 높이면 무조건 좋아질까?)
   - 힌트: 고차항은 값이 커지므로 앞에 `StandardScaler()`를 함께 파이프라인으로 묶는다.
4. 산점도 위에 **1차 직선과 2차 곡선**을 함께 그려 비교한다.
5. **온도 30°C**일 때 수확량을 2차 모델로 **예측**한다.
6. 결과를 **해석**한다. (직선이 왜 부족한가 / 2차가 왜 잘 맞는가 / 차수를 더 높이면? / 적정 차수)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import StandardScaler

plt.rcParams['font.family'] = "Malgun Gothic"
plt.rcParams['axes.unicode_minus'] = False # 마이너스(-) 깨짐방지

df = pd.read_csv('./day02_작물수확.csv')

X = df[['온도']]
y = df['수확량']

# 산점도
plt.figure(figsize=(6, 4))
plt.scatter(X, y, alpha=0.5)
plt.title('재배 온도에 따른 작물 수확량')
plt.xlabel("온도")
plt.ylabel("수확량")
plt.grid()
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 1차
model_1 = LinearRegression()
model_1.fit(X_train, y_train)

pred_1 = model_1.predict(X_test)
r2_1 = r2_score(y_test, pred_1)
rmse_1 = np.sqrt(mean_squared_error(y_test, pred_1))

# 2차
model_2 = make_pipeline(PolynomialFeatures(degree=2, include_bias=False), LinearRegression())
model_2.fit(X_train, y_train)

pred_2 = model_2.predict(X_test)
r2_2 = r2_score(y_test, pred_2)
rmse_2 = np.sqrt(mean_squared_error(y_test, pred_2))

# 출력 결과 비교
print(f"1차 직선 R2 : {r2_1 : .4f}, RMSE : {rmse_1 : .4f}")
print(f"2차 곡선 R2 : {r2_2 : .4f}, RMSE : {rmse_2 : .4f}")

# 차수 추가
degrees = [1, 2, 5, 10]

for deg in degrees :
    model_poly = make_pipeline(
        StandardScaler(),
        PolynomialFeatures(degree=deg, include_bias=False),
        LinearRegression()
    )
    # 학습용, 테스트 예측
    model_poly.fit(X_train, y_train)
    train_pred = model_poly.predict(X_train)
    test_pred = model_poly.predict(X_test)

    # R2 점수 계산
    r2_train = r2_score(y_train, train_pred)
    r2_test = r2_score(y_test, test_pred)

    print(f"차수 {deg:2d} | Train R2: {r2_train:.4f} | Test R2: {r2_test:.4f}")

# 10차 테스트
model_10 = make_pipeline(
    StandardScaler(),
    PolynomialFeatures(degree=10, include_bias=False),
    LinearRegression()
)
model_10.fit(X_train, y_train)

xs = pd.DataFrame({"온도" : np.linspace(X['온도'].min(), X['온도'].max(), 300)})

pred_1_xs = model_1.predict(xs)
pred_2_xs = model_2.predict(xs)
pred_10_xs = model_10.predict(xs)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.5, label="실제 데이터")

plt.plot(xs["온도"], pred_1_xs, color="red", linestyle='--', label="1차 직선")
plt.plot(xs["온도"], pred_2_xs, color="green", linewidth=2, label="2차 곡선")
plt.plot(xs["온도"], pred_10_xs, color="purple", linewidth=1.5, linestyle=":", label="10차(과적합)")

plt.title("1차 직선 vs 2차 곡선 비교 vs 10차비교")
plt.xlabel("온도")
plt.ylabel("수확량")
plt.legend()
plt.show()

# 5단계 예측
sample_30 = pd.DataFrame({'온도' : [30]})
pred_30 = model_2.predict(sample_30)
print(f"온도가 30도 일 때 2차 모델 예상 수확량 : {pred_30[0] : .2f}")

# 직선의 경우 계속 증가하는 형태로 그려지기 때문에 온도에 따른 수확량을 반영하지 못한다
# 2차의 경우 온도가 올라감에 따라 수확량이 늘었다가 고온에서 다시 감소하는 특성을 정확하게 반영하고 있다.
# 차수가 올라가면 학습을 잘되나 평가에서 조금씩 떨어지는 경향을 보여 과대적합 위험이 증가한다.
# 적정 차수는 2차 곡선 모델로 가장 적절함